# Normalization of the chain energies

`energy.e_total` scales every energy term by a power of `linklen` and by a constant `NCONST_*`:

```python
w.conf * NCONST_CONF * e_confinement(pts) / 2 * linklen
w.bend * NCONST_BEND * e_bending(pts)       / linklen
w.frep * NCONST_FREP * e_repulse_far(...)   * linklen**2
```

This notebook derives where both factors come from:

1. **`linklen` powers:** each discrete sum is a Riemann-sum approximation of a line integral along the chain. Rescaling by the right power of the link length makes the result independent of the resolution $N$ (and, with ropelength $L=1$, dimensionless).
2. **`NCONST_*`:** the value of each continuum integral on a ring of unit circumference, in closed form. Dividing by it makes every term equal $1$ on the reference ring, so the weights `w.*` start out balanced.

## Setup and notation

- closed chain of $N$ points $x_i \in \mathbb{R}^3$ on a curve $x(s)$ of length $L$
- uniform link length $h = L/N$ (`ChainProps.linklen`, with $L = 1$)
- sums go to integrals via $\sum_i f(x_i) \approx \frac{1}{h}\int_0^L f(x(s))\,ds$

**Reference ring:** circumference $L$, radius $R = L/2\pi$, curvature $\kappa = 1/R = 2\pi/L$.

In [ ]:
import math
import torch

from geometry_mnist.knotgen.chain import construct_repulsion_mask, calc_distances_squared
from geometry_mnist.knotgen.energy import (
    e_confinement, e_bending, e_repulse_far,
    NCONST_CONF, NCONST_BEND, NCONST_FREP,
)

torch.set_default_dtype(torch.float64)

def ring(n: int, L: float = 1.0) -> torch.Tensor:
    # n points evenly spaced on a circle of circumference L in the xy plane
    R = L / (2 * math.pi)
    t = torch.arange(n) * 2 * math.pi / n
    return torch.stack([R * t.cos(), R * t.sin(), torch.zeros(n)], dim=1)

## 1. Confinement → `/ 2 * linklen`

$$
E_\text{conf} = \sum_i |x_i|^2 \;\approx\; \frac{1}{h}\int_0^L |x(s)|^2\,ds
$$

The raw sum grows like $N$. Multiplying by $h$ gives the integral $\int |x|^2 ds$, which does not depend on resolution and has units of $\text{length}^3$ (fully dimensionless: divide by $L^3$, which is $1$ here). The $\tfrac12$ is the usual $\tfrac12 k r^2$ convention of a harmonic potential, not a unit factor.

**Ring:** $|x| = R$ everywhere, so

$$
\frac12 \int |x|^2 ds = \frac12 R^2 L = \frac{L^3}{8\pi^2} \overset{L=1}{=} 0.012665
$$

## 2. Bending → `/ linklen`

The turning angle at vertex $i$ is $\varphi_i \approx \kappa\,h$, and for small angles $1-\cos\varphi \approx \varphi^2/2$:

$$
E_\text{bend} = \sum_i \big(1-\cos\varphi_i\big) \;\approx\; \sum_i \frac{\kappa_i^2 h^2}{2} \;\approx\; \frac{h}{2}\int_0^L \kappa^2\,ds
$$

The raw sum shrinks like $1/N$. Dividing by $h$ gives $\tfrac12\int\kappa^2 ds$, the classical elastic-rod (Euler–Bernoulli) bending energy, with units of $1/\text{length}$ (dimensionless: multiply by $L$).

**Ring:** $\kappa = 2\pi/L$ everywhere, so

$$
\frac12 \int \kappa^2 ds = \frac12 \cdot \frac{4\pi^2}{L^2}\cdot L = \frac{2\pi^2}{L} \overset{L=1}{=} 19.739
$$

## 3. Far repulsion → `* linklen**2`

This sums over **pairs**, so it becomes a double integral:

$$
E_\text{frep} = \sum_{|i-j|_\text{arc} > \delta} \frac{1}{|x_i - x_j|^2} \;\approx\; \frac{1}{h^2} \iint_{|s-t|_\text{arc} > \delta} \frac{ds\,dt}{|x(s)-x(t)|^2}
$$

Multiplying by $h^2$ gives the double integral, which is **already dimensionless** ($\text{length}^2/\text{length}^2$), so no power of $L$ is needed.

Without the mask the integral diverges at $s=t$ (like $\int d\Delta/\Delta^2$). `construct_repulsion_mask` excludes pairs with arc distance $\le \delta$, and $\delta$ is a **fraction of the chain** (`excl_percentage * n_pts`, default $\delta/L = 0.02$), not a fixed number of points. That is what keeps the term independent of resolution.

**Ring:** the chord between two points at arc distance $\Delta$ is $\frac{L}{\pi}\sin(\pi\Delta/L)$. Integrating $s$ over the ring and $\Delta$ over $(\delta, L-\delta)$:

$$
\iint \frac{ds\,dt}{|x(s)-x(t)|^2}
= L \int_\delta^{L-\delta} \frac{\pi^2}{L^2 \sin^2(\pi\Delta/L)}\,d\Delta
= \pi\Big[-\cot\frac{\pi\Delta}{L}\Big]_\delta^{L-\delta}
= 2\pi\cot\!\Big(\frac{\pi\delta}{L}\Big)
$$

For $\delta/L = 0.02$: $2\pi\cot(0.02\pi) \approx 99.85$. For small $\delta$ this is $\approx 2L/\delta$, so the constant mainly encodes the cutoff.

## Numeric check

Evaluate the three discrete terms on rings of increasing resolution, with the `linklen` rescaling applied (no `NCONST_*`), and compare to the closed forms. The rescaled values should **not** change with $N$ and should converge to the analytic values.

In [ ]:
EXCL = 0.02

analytic = {
    "conf": 1 / (8 * math.pi**2),
    "bend": 2 * math.pi**2,
    "frep": 2 * math.pi / math.tan(math.pi * EXCL),
}
in_code = {"conf": 1 / NCONST_CONF, "bend": 1 / NCONST_BEND, "frep": 1 / NCONST_FREP}

def rescaled_terms(n: int) -> dict[str, float]:
    pts = ring(n)
    h = 1 / n
    mask = construct_repulsion_mask(n, excl_percentage=EXCL)
    dst = calc_distances_squared(pts)
    return {
        "conf": (e_confinement(pts) / 2 * h).item(),
        "bend": (e_bending(pts) / h).item(),
        "frep": (e_repulse_far(dst, mask) * h**2).item(),
    }

print(f"{'N':>6} | {'conf':>10} {'bend':>10} {'frep':>10}")
print("-" * 43)
for n in [128, 256, 512, 1024, 2048, 4096]:
    t = rescaled_terms(n)
    print(f"{n:>6} | {t['conf']:>10.6f} {t['bend']:>10.4f} {t['frep']:>10.3f}")
print("-" * 43)
print(f"{'exact':>6} | {analytic['conf']:>10.6f} {analytic['bend']:>10.4f} {analytic['frep']:>10.3f}")
print(f"{'code':>6} | {in_code['conf']:>10.6f} {in_code['bend']:>10.4f} {in_code['frep']:>10.3f}")

Confinement and bending agree with the closed form to several digits at every $N$. (`NCONST_CONF` uses the rounded `0.0127`, which is 0.3% off the exact $1/8\pi^2$.)

Repulsion stays within a few percent of the closed form but **jitters with $N$** instead of converging smoothly (about 92.7 to 101 above). The cause is the grid: the mask keeps pairs with integer arc distance $> \delta N$, so the effective cutoff jumps between grid points depending on the fractional part of $\delta N$. Because the integrand $\sim 1/\Delta^2$ is steepest right at the cutoff, that shift is visible. The value in the code (`1/99.7`) was measured at $N=1024$, where it happens to sit close to the exact $2\pi\cot(0.02\pi) \approx 99.87$. The closed form is the cleaner reference if it ever needs recomputing, e.g. for a different `excl_percentage`.

## Final normalized terms on the reference ring

With both factors applied, each term should be $\approx 1$ on the unit ring.

In [ ]:
n = 1024
t = rescaled_terms(n)
print(f"conf: {NCONST_CONF * t['conf']:.4f}")
print(f"bend: {NCONST_BEND * t['bend']:.4f}")
print(f"frep: {NCONST_FREP * t['frep']:.4f}")

## Consequence: scaling behaviour

Under a uniform rescaling $x \to \lambda x$ of a fixed shape (continuum terms):

| term | continuum form | scales as | effect |
|---|---|---|---|
| confinement | $\tfrac12\int\lvert x\rvert^2 ds$ | $\lambda^3$ | pulls the chain inward |
| bending | $\tfrac12\int\kappa^2 ds$ | $\lambda^{-1}$ | pushes it outward |
| far repulsion | $\iint \lvert x(s)-x(t)\rvert^{-2} ds\,dt$ | $\lambda^0$ | shape only, no resistance to overall shrinking |

So the equilibrium **size** is set by `w.conf` vs `w.bend`, and `w.frep` only acts on the **shape** (untangling strands).

**Caveat:** `linklen = 1/N` is the nominal link length and assumes ropelength $L \approx 1$. If the chain length drifts during simulation, the hidden powers of $L$ ($L^{-3}$ for confinement, $L$ for bending) start to matter. Then either use `calc_total_ropelength(pts)` in place of $L$, or renormalize the chain to unit length.

In [ ]:
# numeric check of the scaling table: a ring of circumference L instead of 1, with the same N
n = 1024
mask = construct_repulsion_mask(n, excl_percentage=EXCL)

def continuum_terms(L: float) -> tuple[float, float, float]:
    pts = ring(n, L)
    h = L / n  # true link length of this ring
    return (
        (e_confinement(pts) / 2 * h).item(),
        (e_bending(pts) / h).item(),
        (e_repulse_far(calc_distances_squared(pts), mask) * h**2).item(),
    )

base = continuum_terms(1.0)
print(f"{'L':>5} | {'conf':>8} {'bend':>8} {'frep':>8}   ratio to L=1, expected L^3, L^-1, L^0")
for L in [0.5, 1.0, 2.0]:
    ratios = [v / b for v, b in zip(continuum_terms(L), base)]
    print(f"{L:>5} | " + " ".join(f"{r:>8.4f}" for r in ratios))